# Feature Importance (SHAP)

## Set Up

Packages/Libraries

In [ ]:
import sys

sys.path.append("../")
from src.data_utils import get_data, get_models
from src.config import BASE_PATH, SEED

Set globals

In [ ]:
# Data
DATA_DICT = {"base": get_data(is_nomo=False), "nomo": get_data(is_nomo=True)}

# Models
model_dir = BASE_PATH / "models" / "trained"
model_prefix_list = ["lgbm", "xgb", "knn", "svc", "nn", "stack"]

## Base models
model_dict = get_models(model_prefix_list, model_dir)
## Nomogram
model_dict.update(get_models(["lr"], model_dir))


FEAT_ORDER = [
    ##Pre-Op
    "AGE",
    "BMI",
    "SEX",
    "Diabetes",
    "ASA",
    "PRIOREX",
    "PRECT",
    ## Disease
    "RECUR",
    "SITE",
    "SIZE",
    "LYMPH",
    "STAGE",
    "DEFECT",
    "SECONDPRIMARY",
    ## Surg
    "LENGTH",
    "JEWER",
    "OSTEOTOMY",
    "PLATE",
    "FLAP",
    "TRANSFUS",
    "ISCHEMICTIME",
    "OPTIME",
    ## Immediate post-op
    "REOP",
    "LOHS",
    "POSTCT",
    "WOUNDINF",
    "HGB",
    "ALB",
    ## Long term post-op
    "EXPOSURE",
    "MEDUSED",
    "SURGUSED",
    "PLATETIME",
    "FOLLOWTIME",
    ## Misc
    "RADTIME",
]

# RUN SHAP

In [ ]:
import shap, joblib, hashlib, numpy as np
from pathlib import Path

np.random.seed(SEED)
out = Path(BASE_PATH / "shap_artifacts")
out.mkdir(exist_ok=True)
backgrounds = {}

for model_name in model_dict:
    keyword = "nomo" if model_name == "lr" else "base"
    Xb = DATA_DICT[keyword]["X_test"]

    if model_name == "lr":
        # LinearExplainer accepts (mean, cov) — no rows needed at all
        backgrounds[model_name] = (
            Xb.mean(axis=0).values,
            np.cov(Xb.values, rowvar=False),
        )
    else:
        # k weighted centroids, snapped to observed values (keeps one-hots binary)
        k = min(25, len(Xb))
        backgrounds[model_name] = shap.kmeans(Xb, k, round_values=True)

    backgrounds[f"{model_name}__columns"] = list(Xb.columns)

joblib.dump(backgrounds, out / "shap_backgrounds.joblib")
print(hashlib.sha256((out / "shap_backgrounds.joblib").read_bytes()).hexdigest())